According to the DRR paper, "Deep Reinforcement Learning based Recommendation with Explicit User-Item Interactions Modeling"

Popularity recommends the most popular item:
1. the item with the highest average rating 

2. the items with ratings largest number of positive from current available items to the users at each timestep.

They split each user session into 80% training / 20% testing

in offline evaluation the recommender only chooses among items in the current test session because that is the only place with ground-truth feedback.

In [1]:
import os
import numpy as np
import pandas as pd
import random
import pickle
import torch


In [2]:
seed = 316
random.seed(seed)
np.random.seed(seed)

In [3]:
# train_sequences_idx is a Dictionary
# 6040 keys (User indices, NOT User IDs, 0 to 6039)
# Values: list of movie indices, the first 80% of each user's ratings (all ratings 1-5, not filtered for positivity)
with open('../data/train_sequences_idx.pkl', 'rb') as f:
    train_sequences_idx = pickle.load(f)
with open('../data/test_sequences_idx.pkl', 'rb') as f:
    test_sequences_idx = pickle.load(f)

with open('../data/idx2user.pkl', 'rb') as f:
    idx2user = pickle.load(f)
with open('../data/idx2movie.pkl', 'rb') as f:
    idx2movie = pickle.load(f)


# ratings_dict is a Dictionary
# Each key is a tuple (user index, movie index)
# Values are that user's rating for that movie
with open('../data/ratings_dict.pkl', 'rb') as f:
    ratings_dict = pickle.load(f)

with open('../data/movie_index_to_title.pkl', 'rb') as f:
    movie_index_to_title = pickle.load(f)

# Dictionary that maps MovieID to Title for Quick Lookup
with open('../data/movie_id_to_title.pkl', 'rb') as f:
    movie_id_to_title = pickle.load(f)

So for every movie in the train set, we get the average user rating. 

Then, for each user, for each user's test candidate set, we use the average ratings calculated from the train set to recommend the top 5 or 10 movies with the highest average ratings

In [4]:
# Calculate avg. Rating for EACH movie

movie_ratings_lists = {}
# Key = movie_index
# Value = avg. rating

# train_sequences_idx is a Dictionary
# 6040 keys (User indices, NOT User IDs, 0 to 6039)
# Values: list of movie indices, the first 80% of each user's ratings (all ratings 1-5, not filtered for positivity)

for user_index in train_sequences_idx:
    movie_indices = train_sequences_idx[user_index]

    for movie_index in movie_indices:
        rd_key = (user_index, movie_index)
        user_rating = ratings_dict[rd_key]

        if movie_index not in movie_ratings_lists:
            movie_ratings_lists[movie_index] = []
        movie_ratings_lists[movie_index].append(user_rating)

avg_movie_ratings = {}
for movie_index, ratings_list in movie_ratings_lists.items():
    avg_movie_ratings[movie_index] = round(np.mean(ratings_list), 4)

# Sort dictionary by Ratings Greatest to Least
avg_movie_ratings = dict(sorted(avg_movie_ratings.items(), key=lambda x: x[1], reverse=True))

In [5]:
print("Average movie rating of all movies in TRAIN SET")
print(f"# of movies: {len(avg_movie_ratings)}")

print(f"{'ID':<5} | {'Index':<6} | {'Title':<40} | {'Avg Rating':<12}")
print("-" * 70)

for movie_index, avg_rating in avg_movie_ratings.items():
    movie_id = idx2movie[movie_index]
    movie_title = movie_id_to_title[movie_id]
    print(f"{movie_id:<5} | {movie_index:<6} | {movie_title[:40]:<40} | {avg_rating:<12.4f}")

Average movie rating of all movies in TRAIN SET
# of movies: 3667
ID    | Index  | Title                                    | Avg Rating  
----------------------------------------------------------------------
787   | 744    | Gate of Heavenly Peace, The (1995)       | 5.0000      
578   | 564    | Hour of the Pig, The (1993)              | 5.0000      
167   | 161    | Feast of July (1995)                     | 5.0000      
3245  | 3021   | I Am Cuba (Soy Cuba/Ya Kuba) (1964)      | 5.0000      
1830  | 1652   | Follow the Bitch (1998)                  | 5.0000      
53    | 51     | Lamerica (1994)                          | 5.0000      
3233  | 3010   | Smashing Time (1967)                     | 5.0000      
3172  | 2955   | Ulysses (Ulisse) (1954)                  | 5.0000      
2480  | 2287   | Dry Cleaning (Nettoyage à sec) (1997)    | 5.0000      
3888  | 3642   | Skipped Parts (2000)                     | 5.0000      
3905  | 3659   | Specials, The (2000)                     | 

In [6]:
# Save
with open('../data/avg_train_set_movie_ratings.pkl', 'wb') as f:
    pickle.dump(avg_movie_ratings, f)

# Load Average Train Set Movie Ratings


In [7]:
with open('../data/avg_train_set_movie_ratings.pkl', 'rb') as f:
    avg_movie_ratings = pickle.load(f)

In [8]:
# for each user
#     for each user's test candidate set
#         use the average ratings calculated from the train set 
#         recommend the top 5 or 10 movies with the highest average ratings

k = 10  # or 5
print(f"Top {k} movies by avg rating")
print(f"{'Rank':<5} | {'ID':<5} | {'Idx':<6} | {'Title':<40} | {'Avg Rating':<12} | {'Actual User Rating':<12}")
print("-" * 95)

results = {}
for user_index in test_sequences_idx:
    predicted_movie_ratings = {}
    
    movie_indices = train_sequences_idx[user_index]
    for movie_index in movie_indices:
        avg_rating = avg_movie_ratings[movie_index]
        predicted_movie_ratings[movie_index] = avg_rating

    # Sort by average rating (highest first) and get top K
    sorted_movies = sorted(predicted_movie_ratings.items(), key=lambda x: x[1], reverse=True)
    top_k_movies = sorted_movies[:k]
    
    # Store results
    results[user_index] = {
        'top_k_movies': top_k_movies,  # list of (movie_index, avg_rating)
        'movie_indices': [m for m, _ in top_k_movies],
        'avg_ratings': [r for _, r in top_k_movies]
    }

    # Top {k} movies by avg rating
    print(f"User {user_index}:")
    for rank, (movie_idx, avg_rating) in enumerate(top_k_movies, 1):
        movie_id = idx2movie[movie_idx]
        movie_title = movie_index_to_title[movie_idx]
        true_rating = ratings_dict.get((user_index, movie_idx), 'N/A')
        print(f"{rank:<5} | {movie_id:<5} | {movie_idx:<6} | {movie_title[:40]:<40} | {avg_rating:<12.4f} | {true_rating:<12}")
    print("\n")

Top 10 movies by avg rating
Rank  | ID    | Idx    | Title                                    | Avg Rating   | Actual User Rating
-----------------------------------------------------------------------------------------------
User 0:
1     | 527   | 513    | Schindler's List (1993)                  | 4.5097       | 5.0         
2     | 260   | 253    | Star Wars: Episode IV - A New Hope (1977 | 4.4559       | 4.0         
3     | 1207  | 1117   | To Kill a Mockingbird (1962)             | 4.4368       | 4.0         
4     | 720   | 689    | Wallace & Gromit: The Best of Aardman An | 4.4169       | 3.0         
5     | 2762  | 2557   | Sixth Sense, The (1999)                  | 4.4098       | 4.0         
6     | 1193  | 1104   | One Flew Over the Cuckoo's Nest (1975)   | 4.3863       | 5.0         
7     | 2028  | 1848   | Saving Private Ryan (1998)               | 4.3394       | 5.0         
8     | 1197  | 1107   | Princess Bride, The (1987)               | 4.3013       | 3.0        

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [9]:
def compute_metrics_at_k(recommended_items, ground_truth, k):
    """
    Calculate Precision@K and NDCG@K for a single user
    
    Args:
        recommended_items: list of recommended movie indices (in order)
        ground_truth: set of positive movie indices (rating >= 4)
        k: number of recommendations to consider
    
    Returns:
        precision_at_k, ndcg_at_k
    """
    # Take top k recommendations
    top_k = recommended_items[:k]
    
    # Calculate hits (1 if in ground truth, else 0)
    hits = [1 if item in ground_truth else 0 for item in top_k]
    
    # Precision@K
    precision_at_k = sum(hits) / k
    
    # DCG@K
    dcg = sum(hits[i] / np.log2(i + 2) for i in range(len(hits)))
    
    # IDCG@K (ideal: all top k are hits)
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(ground_truth), k)))
    
    # NDCG@K
    ndcg_at_k = dcg / idcg if idcg > 0 else 0.0
    
    return precision_at_k, ndcg_at_k

In [10]:
def get_results(precision, ndcg, top_k_list):
    """
    Average metrics across all users
    
    Args:
        precision: dict with k as key, list of precision values per user
        ndcg: dict with k as key, list of ndcg values per user
        top_k_list: list of k values
    
    Returns:
        results: dict with metric names as keys and average values
    """
    results = {}
    for k in top_k_list:
        results[f'Precision@{k}'] = np.mean(precision[k])
        results[f'NDCG@{k}'] = np.mean(ndcg[k])
    return results

In [11]:
top_k_list = [5, 10]
all_sessions = {}

for user_index in test_sequences_idx:
    test_candidates = test_sequences_idx[user_index]
    
    # Get average ratings for test candidates
    movie_avg_ratings = {m: avg_movie_ratings.get(m, 0.0) for m in test_candidates}
    
    # Sort by average rating (highest first)
    sorted_movies = sorted(movie_avg_ratings.items(), key=lambda x: x[1], reverse=True)
    
    # Get ground truth (positive ratings in test set)
    ground_truth = set()
    for movie_idx in test_candidates:
        rating = ratings_dict.get((user_index, movie_idx), 0)
        if rating >= 4:
            ground_truth.add(movie_idx)
    
    # Store session (for popularity, all 10 recommendations are just the top 10 by avg rating)
    all_sessions[user_index] = {
        'steps': [{'recommended_movie_idx': movie_idx} for movie_idx, _ in sorted_movies[:10]],
        'ground_truth': ground_truth
    }

# Now calculate metrics using your existing code
precision = {k: [] for k in top_k_list}
ndcg = {k: [] for k in top_k_list}

for user_idx, session in all_sessions.items():
    recommended_items = [step['recommended_movie_idx'] for step in session['steps']]
    ground_truth = session['ground_truth']
    
    for k in top_k_list:
        p, n = compute_metrics_at_k(recommended_items, ground_truth, k)
        precision[k].append(p)
        ndcg[k].append(n)

results = get_results(precision, ndcg, top_k_list)
print("\nPopularity Baseline Results:")
for metric, value in results.items():
    print(f"{metric}: {value:.4f}")


# Popularity Baseline Results:
# Precision@5: 0.7319
# Precision@10: 0.6345
    
# NDCG@5: 0.7843
# NDCG@10: 0.8008


Popularity Baseline Results:
Precision@5: 0.7319
NDCG@5: 0.7843
Precision@10: 0.6345
NDCG@10: 0.8008
